# README

- **Author**: `李芸榛（JenniferjjLee）`
- **Created At**: `2026-11-08`
- **Last Modified At**: `2026-11-08`（輸入改用 v7，修正 `捷運科技大樓站(台北教育大學)` 的截斷站名；同日改用 109 站的新版 `stations_in_field` 重跑，臺大人文館改用正式 StationID，並納入羅斯福路三段283巷；同日輸入改讀 v7 Parquet，主檔改為輸出 Parquet，並停止產生範例檔；同日新增 `volume` 欄（借出 + 還入））

---

## What does this file do?

- 從 v7 交易資料（2024、2025）計算**研究範圍內 109 站 × 每小時**的借出量、還入量與淨流量，並依 `trip_group` 拆成各類旅次。範圍 = `stations_in_field.csv`（2026-11-08 版）的 109 站，含 2025 年才啟用的臺大人文館（`500119097`）與羅斯福路三段283巷（`500101265`）。範圍外站點的流量不完整（只收到和台大有往來的旅次），所以不列入。
- **觀測單位**：一個站點的一個小時（`station_id` × `datetime`）。時間格為 2024-01-01 00:00 到 2025-12-31 23:00，共 17,544 小時；109 站 × 17,544 小時 = 1,912,296 列，**沒有借還的小時填 0**。
- **計算方式**：
  - 借出：每筆旅次在**借車站、借車小時**（`rent_time`）計一次，只算範圍內的借車站。
  - 還入：每筆旅次在**還車站、還車小時**（`return_time`）計一次，只算範圍內的還車站。
  - 時間只精確到小時，`datetime` 是該小時的開始時間。
- **依 `trip_group` 拆分**（各類加總 = 總量）：
  - 範圍內站點的借出只可能是 b、c、d：`rent_to_outside`（b 台大借外還）、`rent_to_field`（c 台大借台大還）、`rent_invalid`（d 無效借還）。
  - 範圍內站點的還入只可能是 a、c、d：`return_from_outside`（a 外借台大還）、`return_from_field`（c）、`return_invalid`（d）。
  - `net_flow` = `return_total` − `rent_total`。
  - `volume` = `rent_total` + `return_total`（借還交易量）。
- **`in_service`**：1 表示該站在該小時已啟用。啟用日 = 站位快照的 `first_seen`，和該站第一筆借或還的日期，兩者取較早者；理由是快照從 2024-01-02 才開始，也有缺日，只用 `first_seen` 會晚於實際啟用。臺大人文館從 2025-02-04、羅斯福路三段283巷從 2025-10-10 起啟用（`first_seen` 與第一筆交易日相同）。`in_service = 0` 的列借還都是 0，代表站點尚未啟用，不是沒有人借。
- **放不進時間格的還車**：還車時間超出 2025-12-31 23:00 的 75 筆（2026-01-01 00–07 時的跨年還車，以及 1 筆 2034-08-26 的錯誤紀錄）不計入，所以還入總量比交易資料少 75 筆。

---

## What does this file take?

- **Source Data Sets** (one entry per actual file):
  1. `data/processed/youbike_traffic/YouBike_traffic_2024_v7.parquet`
     - Input key: `traffic_2024`
     - Description: 2024 年台大周邊交易（含 `trip_group`），7,776,981 列 × 10 欄。
     - Documentation: `data/processed/youbike_traffic/YouBike_traffic_doc.ipynb`；產生程式 `data/code/youbike_traffic_station_id_v7.ipynb`
     - SHA-256: `5739637b670bceb34ed2e7271cb78da56f9ea56da310bfb53ce8dccaadd5e119`
  2. `data/processed/youbike_traffic/YouBike_traffic_2025_v7.parquet`
     - Input key: `traffic_2025`
     - Description: 2025 年台大周邊交易，10,253,912 列 × 10 欄。
     - Documentation: 同上
     - SHA-256: `9b1e3beb9d0e4a164c2ad6662764150768452de126be46329566d6030c0c61d9`
  3. `data/processed/stations_in_field.csv`
     - Input key: `stations_in_field`
     - Description: 研究範圍站點清單（2026-11-08 版，109 站），用於 StationID、站名與 `first_seen`。
     - Documentation: pending；產生程式 `data/code/filter_stations_in_field.R`
     - SHA-256: `74dadd7afe58f0b69b0a511f234dc85388e64290d5db420b8426767bad637f9e`

---

## What does this file output?

- `data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.parquet`
  - Output key: `flow`
  - Description: 範圍內 109 站 × 每小時借還流量，14 欄：
    - `station_id`：StationID（9 碼）
    - `station_name`：站名（含 `YouBike2.0_` 前綴）
    - `datetime`：小時開始時間 `YYYY-MM-DD HH:00:00`
    - `in_service`：該站當時是否已啟用（1／0）
    - `rent_total`、`rent_to_outside`、`rent_to_field`、`rent_invalid`：借出總量與 b／c／d 拆分
    - `return_total`、`return_from_outside`、`return_from_field`、`return_invalid`：還入總量與 a／c／d 拆分
    - `net_flow`：還入 − 借出
    - `volume`：借出 + 還入（借還交易量）
  - Role: `analysis`
  - Documentation: `data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_doc.ipynb`
  - Expected schema and row-count rules: 1,912,296 列 × 14 欄；`station_id` × `datetime` 唯一且完整；各類加總等於總量；`volume` = 借出 + 還入；借出總量等於範圍內借車的旅次數，還入總量等於範圍內還車且還車時間在格內的旅次數；`in_service = 0` 的列借還皆為 0。Parquet（zstd）；`station_id`、`station_name`、`datetime` 為字串，其餘 11 欄為 int64。
  - Verification status / SHA-256: 見下方讀回驗證輸出
  - Downstream finding input: `data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.parquet` (same exact file)

All paths are relative to the project root, not the operating-system root.
主檔改存 Parquet（2026-11-08 起），低於 GitHub 單檔 100 MB 上限，可以 commit；先前約 167 MB 的 CSV 版已刪除，檔名仍留在 `data/processed/youbike_station_hourly_flow/.gitignore`。原本的 100 列範例檔已於 2026-11-08 刪除，不再產生。


In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual

In [2]:
# Inputs: the two v7 trip files (with trip_group) and the station list (for StationID and first_seen dates).
IN_DIR = "data/processed/youbike_traffic"
INPUTS = {
    "traffic_2024": (f"{IN_DIR}/YouBike_traffic_2024_v7.parquet", "5739637b670bceb34ed2e7271cb78da56f9ea56da310bfb53ce8dccaadd5e119"),
    "traffic_2025": (f"{IN_DIR}/YouBike_traffic_2025_v7.parquet", "9b1e3beb9d0e4a164c2ad6662764150768452de126be46329566d6030c0c61d9"),
    "stations_in_field": ("data/processed/stations_in_field.csv", "74dadd7afe58f0b69b0a511f234dc85388e64290d5db420b8426767bad637f9e"),
}
OUT_DIR = "data/processed/youbike_station_hourly_flow"
OUTPUT = f"{OUT_DIR}/youbike_station_hourly_flow_2024_2025.parquet"
TRAFFIC_COLUMNS = ["rent_time", "rent_station", "rent_station_id", "return_time", "return_station",
                   "return_station_id", "rent", "bike_type", "infodate", "trip_group"]
OUTSIDE_ID = "-1"
N_STATIONS = 109   # stations_in_field 2026-11-08 version
HOURS = pd.date_range("2024-01-01 00:00", "2025-12-31 23:00", freq="h")
TEXT_READ = {"dtype": str, "keep_default_na": False, "na_filter": False, "encoding": "utf-8-sig"}

input_paths = {}
for key, (rel, expected) in INPUTS.items():
    input_paths[key] = project_file(rel, "processed")
    verify_input_sha256(input_paths[key], expected)
output_path = project_file(OUTPUT, "processed", must_exist=False)
if output_path.exists():
    raise FileExistsError(f"Review the existing output before replacing it: {output_path}")
output_path.parent.mkdir(exist_ok=True)
print(f"{len(INPUTS)} inputs match their expected SHA-256; output paths are free. Hours in the grid: {len(HOURS):,}")

3 inputs match their expected SHA-256; output paths are free. Hours in the grid: 17,544


## 讀入交易與站點

In [3]:
# Read only the columns needed; parse hour stamps.
cols = ["rent_time", "rent_station", "rent_station_id", "return_time", "return_station", "return_station_id", "trip_group"]
trips = pd.concat([pd.read_parquet(input_paths[k], columns=cols) for k in ("traffic_2024", "traffic_2025")],
                  ignore_index=True)
assert len(trips) == 7_776_981 + 10_253_912
assert trips["trip_group"].isin(["a", "b", "c", "d"]).all()
for col in ("rent_time", "return_time"):
    trips[col] = pd.to_datetime(trips[col], format="%Y-%m-%d %H:%M:%S")

# In-field stations: the 109 stations of stations_in_field.
# in_service_from = the earlier of (a) first_seen in the 2024 station snapshots and (b) the date of the station's
# first rental or return. Snapshots start on 2024-01-02 and miss some days, so first_seen alone can be too late.
stations = pd.read_csv(input_paths["stations_in_field"], dtype=str, keep_default_na=False)
assert stations.groupby("StationID")["StationNameZh_tw"].nunique().max() == 1
station_table = (stations.assign(first_seen=pd.to_datetime(stations["first_seen"]))
                 .groupby("StationID", as_index=False)
                 .agg(station_name=("StationNameZh_tw", "first"), first_seen=("first_seen", "min")))
assert len(station_table) == N_STATIONS
station_table = station_table.rename(columns={"StationID": "station_id"})
first_trip = pd.concat([
    trips.loc[trips["rent_station_id"] != OUTSIDE_ID].groupby("rent_station_id")["rent_time"].min(),
    trips.loc[trips["return_station_id"] != OUTSIDE_ID].groupby("return_station_id")["return_time"].min(),
]).groupby(level=0).min().dt.normalize()
station_table["first_trip"] = station_table["station_id"].map(first_trip)
assert station_table["first_trip"].notna().all()
station_table["in_service_from"] = station_table[["first_seen", "first_trip"]].min(axis=1)
station_table = station_table.sort_values("station_id").reset_index(drop=True)
assert len(station_table) == N_STATIONS
print("啟用日晚於 2024-01-01 的站（in_service_from = first_seen 與第一筆交易日較早者）：")
display(station_table[station_table["in_service_from"] > pd.Timestamp("2024-01-01")])
print("第一筆交易早於 first_seen 的站數：", int((station_table["first_trip"] < station_table["first_seen"]).sum()))

啟用日晚於 2024-01-01 的站（in_service_from = first_seen 與第一筆交易日較早者）：


,station_id,station_name,first_seen,first_trip,in_service_from
40,500101234,YouBike2.0_和平東路二段90巷,2024-03-19,2024-03-19,2024-03-19
41,500101240,YouBike2.0_和平東路二段76巷,2024-09-20,2024-09-20,2024-09-20
42,500101250,YouBike2.0_國立臺北教育大學創意館,2024-12-13,2024-12-13,2024-12-13
43,500101265,YouBike2.0_羅斯福路三段283巷,2025-10-10,2025-10-10,2025-10-10
49,500106135,YouBike2.0_羅斯福路四段108巷口,2024-10-15,2024-10-15,2024-10-15
71,500119059,YouBike2.0_臺大共同教室北側,2024-01-02,2024-06-12,2024-01-02
103,500119092,YouBike2.0_臺大管理學院一館,2024-08-13,2024-08-13,2024-08-13
104,500119093,YouBike2.0_臺大禮賢樓西側,2024-08-16,2024-08-16,2024-08-16
105,500119094,YouBike2.0_臺大大一女餐廳廣場,2024-08-13,2024-08-13,2024-08-13
106,500119095,YouBike2.0_臺大學新館,2024-08-13,2024-08-09,2024-08-09


第一筆交易早於 first_seen 的站數： 98


## 計算每站每小時借還流量並寫出

In [4]:
# Rentals are counted at the rental station in the rental hour; returns at the return station in the return hour.
# Only in-field ends are counted (-1 = outside). Rental at an in-field station is group b, c or d; return is a, c or d.
RENT_GROUPS = {"b": "rent_to_outside", "c": "rent_to_field", "d": "rent_invalid"}
RETURN_GROUPS = {"a": "return_from_outside", "c": "return_from_field", "d": "return_invalid"}

rent = trips.loc[trips["rent_station_id"] != OUTSIDE_ID, ["rent_station_id", "rent_time", "trip_group"]]
assert rent["trip_group"].isin(list(RENT_GROUPS)).all(), "An in-field rental has group a."
ret = trips.loc[trips["return_station_id"] != OUTSIDE_ID, ["return_station_id", "return_time", "trip_group"]]
assert ret["trip_group"].isin(list(RETURN_GROUPS)).all(), "An in-field return has group b."

out_of_grid = ret[~ret["return_time"].between(HOURS[0], HOURS[-1])]
print(f"還車時間超出 2024-01-01 00 時～2025-12-31 23 時、無法放進格子的還車：{len(out_of_grid):,} 筆；"
      f"還車時間：{sorted(out_of_grid['return_time'].dt.strftime('%Y-%m-%d %H時').unique().tolist())}")
ret = ret[ret["return_time"].between(HOURS[0], HOURS[-1])]

rent_counts = (rent.groupby(["rent_station_id", "rent_time", "trip_group"]).size().unstack(fill_value=0)
               .reindex(columns=list(RENT_GROUPS), fill_value=0).rename(columns=RENT_GROUPS))
rent_counts.index.names = ["station_id", "datetime"]
ret_counts = (ret.groupby(["return_station_id", "return_time", "trip_group"]).size().unstack(fill_value=0)
              .reindex(columns=list(RETURN_GROUPS), fill_value=0).rename(columns=RETURN_GROUPS))
ret_counts.index.names = ["station_id", "datetime"]

grid = pd.MultiIndex.from_product([station_table["station_id"], HOURS], names=["station_id", "datetime"])
flow = (pd.DataFrame(index=grid).join(rent_counts).join(ret_counts).fillna(0).astype("int64").reset_index())
flow["rent_total"] = flow[list(RENT_GROUPS.values())].sum(axis=1)
flow["return_total"] = flow[list(RETURN_GROUPS.values())].sum(axis=1)
flow["net_flow"] = flow["return_total"] - flow["rent_total"]
flow["volume"] = flow["rent_total"] + flow["return_total"]
flow = flow.merge(station_table[["station_id", "station_name", "in_service_from"]], on="station_id", how="left", validate="many_to_one")
flow["in_service"] = (flow["datetime"] >= flow["in_service_from"]).astype(int)
OUTPUT_COLUMNS = ["station_id", "station_name", "datetime", "in_service",
                  "rent_total", "rent_to_outside", "rent_to_field", "rent_invalid",
                  "return_total", "return_from_outside", "return_from_field", "return_invalid", "net_flow", "volume"]
flow = flow[OUTPUT_COLUMNS].sort_values(["station_id", "datetime"]).reset_index(drop=True)
flow["datetime"] = flow["datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")

output_path.touch(exist_ok=False)  # fails if a file appeared since the setup check
flow.to_parquet(output_path, index=False, compression="zstd")
print(f"寫出 {len(flow):,} 列 × {flow.shape[1]} 欄（{len(station_table)} 站 × {len(HOURS):,} 小時）")


還車時間超出 2024-01-01 00 時～2025-12-31 23 時、無法放進格子的還車：75 筆；還車時間：['2026-01-01 00時', '2026-01-01 01時', '2026-01-01 02時', '2026-01-01 03時', '2026-01-01 06時', '2026-01-01 07時', '2034-08-26 10時']


寫出 1,912,296 列 × 14 欄（109 站 × 17,544 小時）


## 讀回驗證

In [5]:
# Reopen the saved file and check it against the trip data.
saved = pd.read_parquet(output_path)
assert saved.notna().all().all(), "null values in the Parquet file"
assert all(saved[c].dtype == "int64" for c in OUTPUT_COLUMNS[3:]), "count columns must be int64"
count_cols = [c for c in OUTPUT_COLUMNS if c not in ("station_id", "station_name", "datetime", "in_service")]
assert list(saved.columns) == OUTPUT_COLUMNS
assert len(saved) == N_STATIONS * len(HOURS) and not saved.duplicated(["station_id", "datetime"]).any(), "grid incomplete or duplicated"
assert saved["datetime"].nunique() == len(HOURS) and saved["station_id"].nunique() == N_STATIONS
assert (saved[count_cols].drop(columns="net_flow") >= 0).all().all()
assert (saved["rent_total"] == saved[list(RENT_GROUPS.values())].sum(axis=1)).all()
assert (saved["return_total"] == saved[list(RETURN_GROUPS.values())].sum(axis=1)).all()
assert (saved["net_flow"] == saved["return_total"] - saved["rent_total"]).all()
assert (saved["volume"] == saved["rent_total"] + saved["return_total"]).all()
# Totals agree with the trip data.
assert saved["rent_total"].sum() == len(rent) and saved["return_total"].sum() == len(ret)
for g, col in RENT_GROUPS.items():
    assert saved[col].sum() == (rent["trip_group"] == g).sum(), col
for g, col in RETURN_GROUPS.items():
    assert saved[col].sum() == (ret["trip_group"] == g).sum(), col
# No flow before a station is in service.
assert (saved.loc[saved["in_service"] == 0, ["rent_total", "return_total"]] == 0).all().all(), "flow before in_service"
assert sha256_file(input_paths["traffic_2024"]) == INPUTS["traffic_2024"][1]
assert sha256_file(input_paths["traffic_2025"]) == INPUTS["traffic_2025"][1]
totals = saved[count_cols].sum()
print(f"表 1　全部站點與小時的加總（{len(saved):,} 列）")
display(totals.map(lambda x: f"{int(x):,}").to_frame("合計").T)
print({"data_path": OUTPUT, "rows": len(saved), "columns": saved.shape[1], "in_service rows": int(saved["in_service"].sum()),
       "size MB": round(output_path.stat().st_size / 1e6, 1),
       "sha256": sha256_file(output_path), "status": "read-back checks passed"})


表 1　全部站點與小時的加總（1,912,296 列）


,rent_total,rent_to_outside,rent_to_field,rent_invalid,return_total,return_from_outside,return_from_field,return_invalid,net_flow,volume
合計,"14,110,392","4,261,858","9,845,805","2,729","13,768,960","3,920,467","9,845,764","2,729","-341,432","27,879,352"


{'data_path': 'data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.parquet', 'rows': 1912296, 'columns': 14, 'in_service rows': 1836720, 'size MB': 11.4, 'sha256': '3db6469dc42efb8dc7fa24661096de56b7b6fd71465d3fd9f3214767daff41ca', 'status': 'read-back checks passed'}


## 結果摘要

以下依 2026-11-08 由上而下完整執行的實際輸出整理（最後一次是輸入改讀 v7 Parquet、主檔改為輸出 Parquet 後，以 nbclient 在乾淨的 kernel 執行）。輸入是 v7 交易檔（修正 `捷運科技大樓站(台北教育大學)` 的截斷站名）；同日改用 109 站的新版 `stations_in_field` 後，v7 與本檔都重新產生，取代先前臺大人文館用 `-2`、283巷算範圍外的版本。

- **輸入**：3 個輸入檔的 SHA-256 都符合預期（兩個 v7 交易檔共 7,776,981 + 10,253,912 列），處理後兩個交易檔沒有變動。
- **改存 Parquet**（2026-11-08）：主檔改為 Parquet（zstd），9.9 MB，先前的 CSV 版約 167 MB。在本 notebook 之外以 `pd.testing.assert_frame_equal` 比對，與 CSV 版逐格相同（欄位型態也相同：前 3 欄字串、其餘 int64），比對後刪除 CSV 版；原本的範例檔已刪除，不再產生。
- **新增 `volume` 欄**（2026-11-08）：`volume` = `rent_total` + `return_total`，放在最後一欄。重新產生後主檔 11.4 MB；在本 notebook 之外以 `pd.testing.assert_frame_equal` 比對，原有 13 欄與前一版逐格相同。
- **輸出**：`youbike_station_hourly_flow_2024_2025.parquet`，1,912,296 列 × 14 欄（109 站 × 17,544 小時），已重新讀回，所有 assertion 通過。其中 1,836,720 列 `in_service = 1`。
- **加總**（全部站點與小時）：
  - 借出 14,110,392 = 台大借外還 4,261,858 + 台大借台大還 9,845,805 + 無效借還 2,729
  - 還入 13,768,960 = 外借台大還 3,920,467 + 台大借台大還 9,845,764 + 無效借還 2,729
  - 淨流量 −341,432
  - 交易量 `volume` 27,879,352（= 借出 + 還入）
  - 和交易資料比：還入少了 75 筆（外借台大還少 34、台大借台大還少 41），就是還車時間超出 2025-12-31 23:00 的旅次。
- **啟用日**：98 站的第一筆交易早於 `first_seen`，大多是 2024-01-01 當天（快照從 01-02 才開始）。啟用日晚於 2024-01-01 的有 12 站，包括 2024 年陸續新設的站、臺大人文館（2025-02-04）和羅斯福路三段283巷（2025-10-10）。其中 `臺大共同教室北側` 的 `first_seen` 是 2024-01-02，但原始資料到 2024-06 才出現這個站名（第一筆交易 2024-06-12），所以它 2024 年上半年雖然 `in_service = 1`，借還都是 0。
- **改用 109 站清單前後比對**（2026-11-08 在本 notebook 之外另以程式比對，前版臺大人文館的 ID 換成 `500119097` 後對齊）：
  - 新增 `500101265`（羅斯福路三段283巷）17,544 列：借出 7,155、還入 6,833、淨流量 −322，`in_service = 1` 共 1,992 小時。
  - 其他 108 站每小時的 `rent_total`、`return_total`、`net_flow`、`in_service`、站名都與前版相同；只有和 283巷 往來的旅次從範圍外改算範圍內，使 `rent_to_outside`／`rent_to_field`（2,481 個站點小時）與 `return_from_outside`／`return_from_field`（2,598 個站點小時）的拆分改變。
- **與 v6 版本比對**（改用 109 站清單前，2026-11-08 在本 notebook 之外另以程式比對）：
  - 除了 `500101166` 以外，107 站每小時的 `rent_total`、`return_total`、`net_flow`、`in_service` 都與 v6 版相同；只有 `rent_to_outside`／`rent_to_field`、`return_from_outside`／`return_from_field` 的拆分改變（另一端是本站的旅次從 a／b 改歸 c）。
  - `500101166` 在 2024 年：借出 24 → 88,116、還入 27 → 83,182；第一筆交易從 2024-07-07 改為 2024-01-01，`in_service` 多了 24 個小時（2024-01-01 當天）。
  - 與快照 `youbike_flow_hourly_2024.parquet` 的 `rider_net_flow` 比對，本站每小時淨流量的相關係數從 0.03 提高到 0.78。
- **使用注意**：
  - 淨流量只反映騎乘，不含調度車；不能直接當作站上車數的變化。
  - d 組（無效借還）只在 2025-08 以後出現，`rent_invalid`／`return_invalid` 不能做跨期比較。
  - 臺大人文館和羅斯福路三段283巷只在 2025 年啟用，跨年比較需要控制站點數或只用 `in_service = 1` 的列。